In [2]:
# autoload
%load_ext autoreload
%autoreload 2
%matplotlib widget

# load pgl actions
from pgl import pglActions

import numpy as np
import mne

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

Load sesssion runs

---

In [16]:
newSession = pglActions.loadSession.execute()
if newSession: session = newSession

In [17]:
session.saveRawSession()

(pglStorage:saveRawSession) Raw checkpoint 2 is complete and available in pgl.pgl.


2

---

Load Net Station mff file

---

In [ ]:
session = pglActions.loadNetStation.execute(session=session)

---

Align session timing with mne

---

In [ ]:
newSession = pglActions.alignSessionToMNE.execute(session=session, channelName='D255')
if newSession: session = newSession

In [ ]:
catchTrial = 1022
blankTrial = 1023
for iRun, run in enumerate(session.runs):
    run.triggers = []
    task = run.getTask()
    for p in task.data.params:
        imageNums = p['imageNum']
        if imageNums is None:
            imageNums = arr = [blankTrial] * 10
        else:
            imageNums = [i if i > 0 else catchTrial for i in imageNums]
        run.triggers = run.triggers + imageNums
    print(f"{iRun}: (n={len(run.triggers)}) {run.triggers} ")


In [ ]:
newSession = pglActions.alignSegmentsToTriggers.execute(session=session, channelName='D255', segments=[0,2,4,6,8,10,12,14,16,18], outputChannelName='triggers')
if newSession: session = newSession

---

Set names of events

---

In [ ]:
# configure the concatenation
mneConfigureEvents = pglActions.mneConfigureEvents()

# group all the things labels into one
grouped = {
    'things': range(1,200),
    'blank': 1022,
    'catch': 1023
}

# label each stimulus with its own name
each = {
    f"stimulus {code:03d}": code
    for code in range(1, 200)
}
each["blank"] = 1022
each["catch"] = 1023

# label each stimulus with its own name
stimulus = {
    f"stimulus {code:03d}": code
    for code in range(1, 200)
}

triggerLabelSets = {
    "grouped": grouped,
    "each": each,
    "stimulus": stimulus
}

# configure
mneConfigureEvents.configure(triggerChannel='triggers', triggerShortestEvent=1, triggerLabelSets = triggerLabelSets)
#mneConfigureEvents.configure(triggerChannel='D255', triggerShortestEvent=1, triggerLabelSets = triggerLabelSets)

In [ ]:
# configure the events
newSession = mneConfigureEvents.run(session)
if newSession: session = newSession

---

Concatenate

---

In [ ]:
newSession = pglActions.mneConcatenate.execute(session=session, all=True)
if newSession: session = newSession

---

Filter

---

In [ ]:
newSession = pglActions.mneFilter.execute(session=session, lowCutoff=0.5, highCutoff=40.0, notch=True, notchFrequency=60.0)
if newSession: session = newSession

---

Bads handling

---

In [ ]:
newSession = pglActions.mneBads.execute(session=session, excludeWhenPositionIsNaN=True, extraBads=['E11','E22'], interpolationOrigin="zero", method="interpolate")
if newSession: session = newSession

---

Mark inter-runs as bad

---

In [ ]:
newSession = pglActions.mneMarkInterRunsAsBad.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

create epochs

---

In [ ]:
newSession = pglActions.mneCreateEpochs.execute(session=session, tmin=0.0, tmax=2.0)
if newSession: session = newSession

---

Downsample epochs

---

In [ ]:
newSession = pglActions.mneDownsampleEvoked.execute(session=session,downsampleFrequency=200)
if newSession: session = newSession

---

Plot evoked

---

In [ ]:
mnePlotEvoked = pglActions.mnePlotEvoked()
#mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#mnePlotEvoked.configure(set="each",label="stimulus 010",picks='eeg')
#mnePlotEvoked.configure(set="grouped",label="catch",minFreq=0,maxFreq=20,picks='eeg')
mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks='eeg')
#mnePlotEvoked.configure(set="each",label="stimulus 021",minFreq=0,maxFreq=20,picks='eeg')

In [ ]:
newSession = mnePlotEvoked.run(session=session)

---

Plot evoked for each stimulus type

---

In [ ]:
newSession = pglActions.mnePlotEvokedEach.execute(session=session, set="each", picks='eeg')
if newSession: session = newSession

---

Compute NCSNR

---

In [ ]:
#mneComputeNCSNR = pglActions.mneComputeNCSNR()
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag")
#mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="mag",channelName="L213_bz-s115")
mneComputeNCSNR.configure(set="grouped",label="things",conditionIDColumn="code",chType="eeg",channelName="E71")

In [ ]:
newSession = mneComputeNCSNR.run(session)
if newSession: session = newSession

---

Blink detection

---

In [ ]:
# fit and display ICA components
newSession = pglActions.mneICA.execute(session=session, chType='eeg', nComponents=10)
if newSession: session = newSession

In [ ]:
# mark as bad based on ICA component from above and threshold in std
newSession = pglActions.mneMarkBlinks.execute(session=session, component=1, threshold=10)
if newSession: session = newSession

In [ ]:
# Create epochs, respecting the bads from the eyeblink (will fill time periods with NaN)
newSession = pglActions.mneCreateEpochsWithBads.execute(session=session)
if newSession: session = newSession

---

Plot evoked respecting Nans introduced by blink detection

---

In [ ]:
mnePlotEvokedNan = pglActions.mnePlotEvokedNan()
#mnePlotEvoked.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks="L213")
#mnePlotEvoked.configure(set="each",label="stimulus 010",picks='eeg')
#mnePlotEvoked.configure(set="grouped",label="catch",minFreq=0,maxFreq=20,picks='eeg')
mnePlotEvokedNan.configure(set="grouped",label="things",minFreq=0,maxFreq=20,picks='eeg')
#mnePlotEvoked.configure(set="each",label="stimulus 021",minFreq=0,maxFreq=20,picks='eeg')

In [ ]:
newSession = mnePlotEvokedNan.run(session=session)

---

Plot evoked each NaN version

---

In [ ]:
newSession = pglActions.mnePlotEvokedEachNan.execute(session=session, set="each", picks='eeg',topoMin =-25, topoMax=25, timeseriesMin=-25, timeseriesMax=25, topoTime=284)
if newSession: session = newSession

---

Scratch code

---

In [ ]:
dataMatrix = session.mne.getDataMatrix()

In [ ]:
dataMatrix.print()

In [ ]:
trialIndexes = [coordName for coordName, coord in dataMatrix.coords.items() if coord.dims == ("trial",)]
print(trialIndexes)

In [ ]:
occipitalSensors = ["E70", "E71", "E74", "E75", "E76", "E82", "E83"]
trialData = dataMatrix.where(dataMatrix.coords["stimulus"].notnull(), drop=True).sel(sensor=occipitalSensors)
stimulusTypes = np.unique(trialData.coords["stimulus"].values)

# z-score individual trials separately for each sensor and time point
trialMean = trialData.mean(dim="trial")
trialSTD = trialData.std(dim="trial", ddof=1)
zscoredData = (trialData - trialMean) / trialSTD.where(trialSTD > 0)

# initialize arrays
noiseVariances = []

for stimulusType in stimulusTypes:
    # get the trial data which match these conditions
    stimulusTypeData = zscoredData.where(zscoredData.coords["stimulus"] == stimulusType, drop=True)
    # estimate the within-stimulus variance using n-1
    noiseVariances.append(stimulusTypeData.var(dim="trial", ddof=1))

# average noise variance across stimulus types, keeping sensors separate
noiseVariance = np.mean(np.stack(noiseVariances), axis=0)

# estimate signal variance from total z-scored variance of 1
signalVariance = 1 - noiseVariance

# compute NSD-style NCSNR separately for each sensor and time point
varianceRatio = np.divide(np.maximum(signalVariance, 0), noiseVariance, out=np.full_like(noiseVariance, np.nan), where=noiseVariance > 0)
ncsnrBySensor = np.sqrt(varianceRatio)

# pool z-scored variances across sensors for an occipital ROI estimate
roiNoiseVariance = noiseVariance.mean(axis=0)
roiSignalVariance = 1 - roiNoiseVariance
roiVarianceRatio = np.divide(np.maximum(roiSignalVariance, 0), roiNoiseVariance, out=np.full_like(roiNoiseVariance, np.nan), where=roiNoiseVariance > 0)
roiNcsnr = np.sqrt(roiVarianceRatio)

In [ ]:
import matplotlib.pyplot as plt

timeValues = trialData.coords["time"].values
sensorNames = trialData.coords["sensor"].values
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# plot NCSNR for each sensor
for sensorIndex, sensorName in enumerate(sensorNames):
    axes[0].plot(timeValues, ncsnrBySensor[sensorIndex], label=sensorName)
axes[0].set_ylabel("NCSNR")
axes[0].set_title("NCSNR by sensor")
axes[0].legend(ncol=len(sensorNames), fontsize=9)

# plot NCSNR from pooled z-scored variances across sensors
axes[1].plot(timeValues, roiNcsnr, color="black")
axes[1].set_ylabel("NCSNR")
axes[1].set_title("Occipital ROI NCSNR")
axes[1].set_xlabel("Time (s)")

for ax in axes:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
occipitalSensors = ["E70", "E71", "E74", "E75", "E76", "E82", "E83"]
stimulusTypes = np.unique(dataMatrix.coords["stimulus"].dropna(dim="trial").values)

# initialize arrays
meanStimulusTypeData = []
stdStimulusTypeData = []
trialCounts = []

for stimulusType in stimulusTypes:
    # get the trial data which match these conditions
    stimulusTypeData = dataMatrix.where(dataMatrix.coords["stimulus"] == stimulusType, drop=True).sel(sensor=occipitalSensors)
    # compute the mean
    meanStimulusTypeData.append(stimulusTypeData.mean(dim="trial"))
    # estimate the standard deviation, using unbiased estimator - i.e. n-1
    stdStimulusTypeData.append(stimulusTypeData.std(dim="trial", ddof=1))
    # count valid trials at each sensor and time point
    trialCounts.append(stimulusTypeData.count(dim="trial"))

# stack as stimulus type x sensor x time
stimulusMeans = np.stack(meanStimulusTypeData)
stimulusStds = np.stack(stdStimulusTypeData)
trialCounts = np.stack(trialCounts)

# estimate variance across stimulus means and residual noise from averaging
observedStimulusVariance = np.var(stimulusMeans, axis=0, ddof=1)
residualNoiseVariance = np.mean(stimulusStds**2 / trialCounts, axis=0)

# subtract noise variance before taking the square root
correctedStimulusVariance = observedStimulusVariance - residualNoiseVariance
stimulusSTD = np.sqrt(np.maximum(correctedStimulusVariance, 0)).mean(axis=0)

# average within-stimulus standard deviations across stimulus types and sensors
noiseSTD = np.mean(stimulusStds, axis=(0, 1))

# compute the ratio at each time point
stdRatio = np.divide(stimulusSTD, noiseSTD, out=np.full_like(stimulusSTD, np.nan), where=noiseSTD > 0)

In [ ]:
import matplotlib.pyplot as plt

timeValues = dataMatrix.coords["time"].values

# average across stimulus types, keeping sensors separate
meanSensorData = stimulusMeans.mean(axis=0)
stdSensorData = stimulusStds.mean(axis=0)

# create five rows with a shared time axis
fig, axes = plt.subplots(5, 1, figsize=(12, 14), sharex=True)

# plot the mean response for each sensor
for sensorIndex, sensorName in enumerate(occipitalSensors):
    axes[0].plot(timeValues, meanSensorData[sensorIndex] * 1e6, label=sensorName)
axes[0].set_ylabel("Mean (µV)")
axes[0].set_title("Mean response by sensor")
axes[0].legend(ncol=len(occipitalSensors), fontsize=9)

# plot the mean within-stimulus standard deviation for each sensor
for sensorIndex, sensorName in enumerate(occipitalSensors):
    axes[1].plot(timeValues, stdSensorData[sensorIndex] * 1e6, label=sensorName)
axes[1].set_ylabel("SD (µV)")
axes[1].set_title("Within-stimulus SD by sensor")

# plot the noise-corrected signal standard deviation
axes[2].plot(timeValues, stimulusSTD * 1e6)
axes[2].set_ylabel("Signal SD (µV)")
axes[2].set_title("Noise-corrected signal SD, averaged across sensors")

# plot the noise standard deviation
axes[3].plot(timeValues, noiseSTD * 1e6)
axes[3].set_ylabel("Noise SD (µV)")
axes[3].set_title("Noise SD, averaged across stimulus types and sensors")

# plot the signal/noise standard deviation ratio
axes[4].plot(timeValues, stdRatio)
axes[4].set_ylabel("Signal / noise")
axes[4].set_title("Signal SD / noise SD")
axes[4].set_xlabel("Time (s)")

for ax in axes:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
print(session.mne.epochs.baseline)

In [ ]:
"""Blink detection from a hand-chosen ICA topography.

Workflow
  1. fit_blink_ica()            ICA on the sensors (once; slow)
  2. plot_top_components()      look at the topographies, pick one by eye
  3. make_blink_channel()       weight the sensors by that topography -> one 'eog' channel
  4. detect_blinks()            threshold + refractory period -> events, windows, annotations
  5. plot_blinks()              overlay of all events + full-duration trace with window bars
  6. mark_blinks_bad(), make_epochs(), average_nan()   BAD annotations -> epochs -> NaN-aware averages

Steps 3-5 are fast, so iterate on THRESH_Z / MIN_SEPARATION_S / PRE_S / POST_S freely.
"""

from types import SimpleNamespace
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
from scipy.signal import find_peaks

# --------------------------------------------------------------------------- #
# Parameters (hand-set)
# --------------------------------------------------------------------------- #
# ICA fit
PICKS = "eeg"            # anything raw.pick() accepts: 'meg', 'mag', 'eeg', ...
N_COMPONENTS = 20
N_SHOW = 10              # how many topographies to display
FIT_HIGHPASS_HZ = 1.0    # ICA is fit on a 1 Hz high-passed copy; raw is untouched
FIT_DECIM = 5
RANDOM_STATE = 42

# Blink channel / detection
COMPONENT = None         # <- set by hand after looking at the topographies
BAND_HZ = (1.0, 10.0)    # band-pass applied to the weighted channel; None to skip
THRESH_Z = 5.0           # event criterion, in robust SDs (median/MAD) of the filtered channel
POLARITY = "auto"        # 'auto' | 'pos' | 'neg' (ICA sign is arbitrary; blinks are made positive)
MIN_SEPARATION_S = 0.5   # bounce guard: peaks closer than this collapse to the largest one
PRE_S = 0.15             # window bar starts this long before the event (peak)
POST_S = 0.35            # window bar ends this long after the event (peak)
PLOT_PAD_S = 0.25        # extra context shown either side of the window in the overlay plot
EVENT_ID = 998
ANNOTATION_LABEL = "BAD_blink"


# --------------------------------------------------------------------------- #
# 1-2. ICA and topographies
# --------------------------------------------------------------------------- #
def fit_blink_ica(
    raw,
    picks=PICKS,
    n_components=N_COMPONENTS,
    highpass=FIT_HIGHPASS_HZ,
    decim=FIT_DECIM,
    random_state=RANDOM_STATE,
):
    """Fit FastICA on a high-passed copy of the picked, non-bad channels."""
    fit_raw = raw.copy().pick(picks, exclude="bads").load_data()
    ch_types = set(fit_raw.get_channel_types())
    if len(ch_types) > 1:
        warnings.warn(
            f"PICKS gives several channel types {sorted(ch_types)}. Topography "
            "weights from different types are in different units; pick one type "
            "(e.g. 'mag') for the blink channel."
        )
    fit_raw.filter(l_freq=highpass, h_freq=None, verbose=False)
    ica = mne.preprocessing.ICA(
        n_components=min(n_components, len(fit_raw.ch_names) - 1),
        method="fastica",
        random_state=random_state,
        max_iter="auto",
    )
    ica.fit(fit_raw, decim=decim, reject_by_annotation=True, verbose=False)
    return ica


def plot_top_components(ica, n=N_SHOW):
    """Topomaps of the first n components (MNE orders them by explained variance)."""
    return ica.plot_components(picks=list(range(min(n, ica.n_components_))), show=False)


# --------------------------------------------------------------------------- #
# 3. Weighted-sensor 'eog' channel
# --------------------------------------------------------------------------- #
def make_blink_channel(
    raw,
    ica,
    component,
    band=BAND_HZ,
    polarity=POLARITY,
    name="ICA-Blink",
):
    """Collapse the sensors to one channel using one component's topography as weights.

    weights = topo / (topo . topo), so a signal that has exactly that topography
    comes out at its own amplitude (a least-squares estimate of that component's
    contribution). Detection runs on a band-passed, robustly z-scored copy.
    """
    if component is None:
        raise ValueError("Set COMPONENT after looking at the topographies.")
    topo = ica.get_components()[:, component]
    weights = topo / np.sum(topo**2)

    data = raw.get_data(picks=ica.ch_names)          # same channels, same order as the ICA
    combo = weights @ data
    combo -= combo.mean()
    del data

    sfreq = raw.info["sfreq"]
    if band is None:
        filt = combo.copy()
    else:
        filt = mne.filter.filter_data(combo, sfreq, band[0], band[1], verbose=False)

    center = np.median(filt)
    scale = 1.4826 * np.median(np.abs(filt - center))   # robust SD
    z = (filt - center) / scale

    if polarity == "auto":
        sign = 1.0 if np.percentile(z, 99.9) >= -np.percentile(z, 0.1) else -1.0
    elif polarity == "pos":
        sign = 1.0
    elif polarity == "neg":
        sign = -1.0
    else:
        raise ValueError("polarity must be 'auto', 'pos' or 'neg'")
    combo, filt, z = sign * combo, sign * filt, sign * z

    # A real MNE 'eog' channel (broadband, before z-scoring) for use with other
    # MNE tools: raw.add_channels([sig.eog_raw]); mne.preprocessing.create_eog_epochs(...)
    eog_raw = mne.io.RawArray(
        combo[np.newaxis],
        mne.create_info([name], sfreq, ch_types="eog"),
        first_samp=raw.first_samp,
        verbose=False,
    )
    eog_raw.set_meas_date(raw.info["meas_date"])

    return SimpleNamespace(
        eog_raw=eog_raw, z=z, sfreq=sfreq, sign=sign, component=component,
        robust_sd=scale, first_samp=raw.first_samp, first_time=raw.first_time,
        meas_date=raw.info["meas_date"], band=band,
    )


# --------------------------------------------------------------------------- #
# 4. Event detection
# --------------------------------------------------------------------------- #
def detect_blinks(
    sig,
    thresh=THRESH_Z,
    min_separation=MIN_SEPARATION_S,
    pre=PRE_S,
    post=POST_S,
    event_id=EVENT_ID,
    label=ANNOTATION_LABEL,
):
    """Peaks above `thresh`; peaks within `min_separation` of a larger one are dropped.

    The event time is the peak. Each event gets a window [peak - pre, peak + post].
    Noise that makes the trace dip below threshold and re-cross within
    `min_separation` therefore yields one event, not several.
    """
    sfreq, z = sig.sfreq, sig.z
    peaks, _ = find_peaks(
        z, height=thresh, distance=max(1, int(round(min_separation * sfreq)))
    )
    times = peaks / sfreq                                  # s from first sample
    dur = len(z) / sfreq
    windows = np.c_[np.clip(times - pre, 0, None), np.clip(times + post, None, dur)]

    events = np.c_[peaks + sig.first_samp, np.zeros_like(peaks), np.full_like(peaks, event_id)]

    # Annotations in the convention MNE expects for this raw's meas_date
    offset = sig.first_time if sig.meas_date is not None else 0.0
    annotations = mne.Annotations(
        onset=windows[:, 0] + offset,
        duration=windows[:, 1] - windows[:, 0],
        description=[label] * len(peaks),
        orig_time=sig.meas_date,
    )
    return SimpleNamespace(
        peaks=peaks, times=times, windows=windows, events=events,
        annotations=annotations, thresh=thresh, pre=pre, post=post,
    )


# --------------------------------------------------------------------------- #
# 5. Plots
# --------------------------------------------------------------------------- #
def plot_blinks(sig, det, pad=PLOT_PAD_S):
    """Fig 1: every event overlaid (peak-aligned). Fig 2: whole recording with window bars."""
    z, sfreq = sig.z, sig.sfreq
    n_ev = len(det.peaks)
    unit = "robust SD"

    # --- overlay of all events
    lo = -int(round((det.pre + pad) * sfreq))
    hi = int(round((det.post + pad) * sfreq))
    rel = np.arange(lo, hi + 1)
    tvec = rel / sfreq
    stack = np.full((n_ev, len(rel)), np.nan)
    for i, p in enumerate(det.peaks):
        idx = p + rel
        ok = (idx >= 0) & (idx < len(z))
        stack[i, ok] = z[idx[ok]]

    fig1, ax1 = plt.subplots(figsize=(7, 4.5))
    ax1.axvspan(-det.pre, det.post, color="tab:red", alpha=0.12, lw=0, label="window")
    if n_ev:
        ax1.plot(tvec, stack.T, color="0.45", lw=0.6, alpha=0.5)
        ax1.plot(tvec, np.nanmean(stack, axis=0), color="k", lw=2, label="mean")
    ax1.axhline(det.thresh, color="tab:red", ls="--", lw=1, label=f"threshold ({det.thresh:g})")
    ax1.axvline(0, color="0.7", lw=0.8)
    ax1.set(xlabel="Time from event peak (s)", ylabel=f"Blink channel ({unit})",
            title=f"{n_ev} events — component {sig.component}")
    ax1.legend(frameon=False)
    ax1.spines[["top", "right"]].set_visible(False)
    fig1.tight_layout()

    # --- whole-duration trace; fill_between merges overlapping windows
    t = np.arange(len(z)) / sfreq
    mask = np.zeros(len(z), bool)
    for start, stop in det.windows:
        mask[int(round(start * sfreq)): int(round(stop * sfreq)) + 1] = True

    fig2, ax2 = plt.subplots(figsize=(14, 4))
    ax2.fill_between(t, 0, 1, where=mask, transform=ax2.get_xaxis_transform(),
                     color="tab:red", alpha=0.25, lw=0, label="blink window")
    ax2.plot(t, z, color="k", lw=0.5)
    ax2.plot(det.times, z[det.peaks], "v", color="tab:red", ms=4, label="event (peak)")
    ax2.axhline(det.thresh, color="tab:red", ls="--", lw=1, label=f"threshold ({det.thresh:g})")
    ax2.set(xlabel="Time (s)", ylabel=f"Blink channel ({unit})",
            title=f"Component {sig.component}: {n_ev} events, "
                  f"{100 * mask.mean():.1f}% of data inside windows",
            xlim=(0, t[-1]))
    ax2.legend(frameon=False, loc="upper right", ncol=3)
    ax2.spines[["top", "right"]].set_visible(False)
    fig2.tight_layout()
    return fig1, fig2


# --------------------------------------------------------------------------- #
# 6. Mark blinks as bad, epoch, and average without them
# --------------------------------------------------------------------------- #
def _merge_windows(windows):
    """Union of overlapping [start, stop] windows (seconds)."""
    if len(windows) == 0:
        return np.empty((0, 2))
    windows = windows[np.argsort(windows[:, 0])]
    merged = [list(windows[0])]
    for start, stop in windows[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], stop)
        else:
            merged.append([start, stop])
    return np.asarray(merged)


def mark_blinks_bad(raw, det, label=ANNOTATION_LABEL, merge=True, copy=False):
    """Add the blink windows to raw.annotations as `label` (must start with 'BAD').

    Idempotent: previous annotations with the same label are replaced, so you can
    re-run after changing the detection parameters. Overlapping windows are merged.
    Other annotations are left alone. Modifies `raw` in place unless copy=True.
    """
    if not label.upper().startswith("BAD"):
        raise ValueError("label must start with 'BAD' for MNE to treat it as bad data")
    raw = raw.copy() if copy else raw
    windows = _merge_windows(det.windows) if merge else det.windows
    meas_date = raw.info["meas_date"]
    offset = raw.first_time if meas_date is not None else 0.0
    new = mne.Annotations(
        onset=windows[:, 0] + offset,
        duration=windows[:, 1] - windows[:, 0],
        description=[label] * len(windows),
        orig_time=meas_date,
    )
    old = raw.annotations.copy()
    old.delete(np.flatnonzero(old.description == label))
    raw.set_annotations(old + new)
    return raw


def bad_sample_mask(raw):
    """Boolean (n_times,) of samples inside ANY 'BAD*' annotation, using MNE's own logic."""
    return np.isnan(raw.get_data(picks=[0], reject_by_annotation="NaN")[0])


def make_epochs(
    raw,
    events,
    event_id,
    tmin,
    tmax,
    mode="nan",
    baseline=(None, 0),
    max_bad_fraction=None,
    **epochs_kw,
):
    """Epoch `raw`, handling samples inside BAD annotations (e.g. blinks).

    mode='drop' : MNE default. Any epoch overlapping a BAD annotation is dropped whole
                  (drop_log reason 'BAD_blink'). Baseline is MNE's.
    mode='nan'  : keep every epoch but set samples inside BAD annotations to NaN
                  (all channels). Baseline is the nan-mean over the baseline window,
                  computed AFTER masking, so blink samples never enter it. Use np.nan*
                  functions or average_nan() afterwards.
                  max_bad_fraction: also drop epochs with more than this fraction of
                  samples bad (e.g. 0.3). Epochs whose whole baseline window is bad are
                  always dropped, since they cannot be baseline-corrected.

    Do all filtering/ICA/resampling BEFORE this: NaNs spread through any filter.
    Note epochs.baseline will read None in 'nan' mode even though a baseline was applied.
    """
    if mode == "drop":
        return mne.Epochs(raw, events, event_id, tmin, tmax, baseline=baseline,
                          preload=True, reject_by_annotation=True, **epochs_kw)
    if mode != "nan":
        raise ValueError("mode must be 'drop' or 'nan'")

    epochs = mne.Epochs(raw, events, event_id, tmin, tmax, baseline=None, preload=True,
                        reject_by_annotation=False, **epochs_kw)
    if not np.isclose(epochs.info["sfreq"], raw.info["sfreq"]):
        raise ValueError("decimated epochs are not supported in mode='nan'")

    # per-epoch bad-sample mask, from MNE's own annotation logic
    sfreq = epochs.info["sfreq"]
    n_times = len(epochs.times)
    start = int(round(epochs.tmin * sfreq))
    first = epochs.events[:, 0] - raw.first_samp + start
    bad = bad_sample_mask(raw)[first[:, None] + np.arange(n_times)]      # (n_epochs, n_times)

    if baseline is not None:
        b0 = epochs.times[0] if baseline[0] is None else baseline[0]
        b1 = epochs.times[-1] if baseline[1] is None else baseline[1]
        in_base = (epochs.times >= b0 - 1e-9) & (epochs.times <= b1 + 1e-9)
        base_dead = bad[:, in_base].all(axis=1)
    else:
        base_dead = np.zeros(len(epochs), bool)
    too_bad = (bad.mean(axis=1) > max_bad_fraction) if max_bad_fraction is not None \
        else np.zeros(len(epochs), bool)

    n_before = len(epochs)
    reasons = np.where(too_bad, "BAD_blink_fraction", "BAD_blink_baseline")
    drop = np.flatnonzero(too_bad | base_dead)
    if len(drop):
        epochs.drop(drop, reason=list(reasons[drop]), verbose=False)
    keep = ~(too_bad | base_dead)
    bad = bad[keep]

    data = epochs.get_data(copy=False)       # view on the preloaded array
    for i in range(len(epochs)):
        data[i][:, bad[i]] = np.nan
    if baseline is not None:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            data -= np.nanmean(data[:, :, in_base], axis=2, keepdims=True)

    print(f"make_epochs[nan]: kept {len(epochs)}/{n_before} epochs; "
          f"{100 * bad.mean():.1f}% of kept samples set to NaN; "
          f"{int((bad.any(axis=1)).sum())} epochs contain some NaN")
    return epochs


def average_nan(epochs, method="mean"):
    """NaN-aware Evoked ('mean' or 'median') plus n_valid, the epoch count at each time point.

    evoked.nave is set to the median of n_valid. For std/sem etc. use the arrays
    directly: np.nanstd(epochs.get_data(), axis=0), divided by sqrt(n_valid) for sem.
    """
    fun = {"mean": np.nanmean, "median": np.nanmedian}[method]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)       # all-NaN time points
        evoked = epochs.average(method=lambda d: fun(d, axis=0))
    n_valid = np.isfinite(epochs.get_data(copy=False)).any(axis=1).sum(axis=0)
    evoked.nave = max(1, int(np.median(n_valid)))
    return evoked, n_valid




In [ ]:

# --------------------------------------------------------------------------- #
# Example (run as cells)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    raw = session.mne.raw  # noqa: F821  (your session object)

    # %% 1-2: fit once, look at topographies
    ica = fit_blink_ica(raw)
    plot_top_components(ica)
    plt.show()
                       # real 'eog' channel in raw

In [ ]:

    # %% 3-5: set COMPONENT by hand, then iterate on the parameters
    COMPONENT = 1
    sig = make_blink_channel(raw, ica, COMPONENT)
    det = detect_blinks(sig, thresh=5.0, min_separation=0.5, pre=0.15, post=0.35)
    plot_blinks(sig, det)
    plt.show()

    # %% optional: use the result
    # raw.set_annotations(raw.annotations + det.annotations)   # epochs reject BAD_* by default
    # raw.add_channels([sig.eog_raw])    

In [ ]:

    # %% 6: mark as bad, epoch, average
    mark_blinks_bad(raw, det)                     # re-run freely; replaces old BAD_blink
    # raw.add_channels([sig.eog_raw])             # optional: real 'eog' channel in raw
    # events, event_id = ...                      # your own events
    epochs = make_epochs(raw, events, event_id, tmin=-0.2, tmax=1.0,
                         mode="nan", baseline=None, max_bad_fraction=0.3)
    evoked, n_valid = average_nan(epochs, "mean")           # or "median"
    data = epochs.get_data()                                # NaN where blinks were
    sd = np.nanstd(data, axis=0)
    # epochs_drop = make_epochs(raw, events, event_id, -0.2, 1.0, mode="drop")  # MNE default

In [ ]:
session.mne.report.save("/Users/justin/Desktop/test.html", overwrite=True, open_browser=True)

In [ ]:
session.runs[0]